In [0]:
# Limpieza de widgets previos al inicio del proceso para evitar conflictos con ejecuciones anteriores
dbutils.widgets.removeAll()

In [0]:
# Importación de librerías de PySpark necesarias para transformaciones de columnas y definición de esquemas
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Definición de parámetros de entrada como widgets parametrizables
# Permite la ejecución dinámica desde Jobs sin modificar el código fuente
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "cat_ftr_smartdata_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsftrsmardatadev01")

In [0]:
# Recuperación de los valores de los widgets para usarlos como variables en el proceso
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
# Construcción de la ruta base en Azure Data Lake Storage Gen2 y ruta al archivo fuente CSV
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
items = ruta + "items.csv"


In [0]:
# Definición explícita del esquema del archivo fuente para garantizar tipado correcto
# y evitar la inferencia automática, que puede generar inconsistencias en el tipo de datos
producto_schema = StructType(fields=[StructField("ID_ITEM", StringType(), False),
                                  StructField("PRODUCT", StringType(), False),
                                  StructField("CATEGORY", StringType(), False)
                                  ])


In [0]:
%python
# Lectura del archivo CSV fuente desde ADLS
df_productos = (spark.read.format("csv")
    .option("header", "true")    
    .schema(producto_schema)
    .load(items))

In [0]:
#display(df_productos)

In [0]:
# Enriquecimiento con campos de auditoría
# - INGESTION_DATE    : fecha y hora exacta en que el registro fue cargado
# - SOURCE_FILE_NAME  : ruta del archivo fuente (trazabilidad de origen)
# - INGESTION_USER    : usuario o service principal que ejecutó la ingesta
df_productos_ingestion_date = (df_productos
    .withColumn("INGESTION_DATE", current_timestamp())
    .withColumn("SOURCE_FILE_NAME", col("_metadata.file_path"))
    .withColumn("INGESTION_USER", current_user())
)

In [0]:
# Selección y orden de columnas finales: campos de negocio seguidos de campos de auditoría
df_productos_final = df_productos_ingestion_date.select(
                                    "ID_ITEM",
                                    "PRODUCT",
                                    "CATEGORY",
                                    "INGESTION_DATE",
                                    "SOURCE_FILE_NAME",
                                    "INGESTION_USER"
                                )


In [0]:
# Escritura en la tabla Delta de la capa bronze con modo overwrite (carga completa)
# overwriteSchema=true permite actualizar el esquema si se agregan nuevas columnas
df_productos_final.write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalogo}.{esquema}.productos")

In [0]:
%sql
-- Verificación de los datos cargados en la tabla bronze, incluyendo campos de auditoría
--select * from cat_ftr_smartdata_dev.bronze.productos limit 10